# 11. Математика глубокого обучения и трансформеров

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/11_deep_learning.ipynb)

Код из раздела [modules/11_deep_learning.md](https://github.com/justxor/math-for-ai-2026/blob/main/modules/11_deep_learning.md#m11). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### Блок трансформера (pre-norm, как в LLaMA)

In [ ]:
import torch, torch.nn.functional as F

def causal_self_attention(x, Wq, Wk, Wv, Wo, n_heads):
    B, T, D = x.shape; hd = D // n_heads
    q, k, v = (t.view(B, T, n_heads, hd).transpose(1, 2) for t in (x @ Wq, x @ Wk, x @ Wv))
    att = q @ k.transpose(-2, -1) / hd ** 0.5                           # (B, h, T, T)
    att = att.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool), 1), float("-inf"))
    y = F.softmax(att, dim=-1) @ v                                      # (B, h, T, hd)
    return y.transpose(1, 2).reshape(B, T, D) @ Wo

### 🏋️ Практика модуля

**11.1.** Проверьте, что ваша реализация attention совпадает с `F.scaled_dot_product_attention`.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
B, T, D, h = 2, 5, 64, 4
x = torch.randn(B, T, D); Wq, Wk, Wv, Wo = [torch.randn(D, D) / 8 for _ in range(4)]
mine = causal_self_attention(x, Wq, Wk, Wv, Wo, h)
q, k, v = ((x @ W).view(B, T, h, D // h).transpose(1, 2) for W in (Wq, Wk, Wv))
ref = F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, D) @ Wo
print(torch.allclose(mine, ref, atol=1e-5))   # True

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def attention():
    rng = np.random.default_rng(7)
    tokens = ["Кот", "сел", "на", "ковёр", ",", "потому", "что", "он", "устал"]
    n, d = len(tokens), 16
    E = rng.normal(size=(n, d))
    E[7] = 0.85 * E[0] + 0.3 * rng.normal(size=d)  # «он» похож на «Кот»
    Wq = np.eye(d) + 0.1 * rng.normal(size=(d, d)); Wk = np.eye(d) + 0.1 * rng.normal(size=(d, d))
    S = (E @ Wq) @ (E @ Wk).T / np.sqrt(d)
    S = np.where(np.tril(np.ones((n, n))) == 1, S, -np.inf)
    A = np.exp(S - S.max(1, keepdims=True)); A /= A.sum(1, keepdims=True)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(A, cmap="Blues")
    ax.set_xticks(range(n), tokens, rotation=45); ax.set_yticks(range(n), tokens)
    ax.set_xlabel("key (на кого смотрим)"); ax.set_ylabel("query (кто смотрит)")
    ax.set_title("Causal self-attention: softmax(QKᵀ/√d + маска)\nкаждая строка — распределение, сумма = 1")
    ax.grid(False); fig.colorbar(im, ax=ax, fraction=0.046)
    save(fig, "attention")

attention()